# Agentic RAG & Multi-Agent Intelligent Workflows
## HR Policy Email Triage Assistant using Agnos/Agno + Zapier

**Use case:** An HR Policy Email Triage Assistant that retrieves company policy information, reasons over employee requests, and triggers workflow actions such as drafting replies, logging requests, creating approval tasks, or escalating urgent security issues.

**Core idea:** Standard LLMs generate text. This system retrieves evidence first, uses specialized agents to reason over the evidence, and then routes the request to a business workflow.


## HR Policy Email Triage Assistant using Agnos/Agno + Zapier

**Use case:** An HR Policy Email Triage Assistant that retrieves company policy information, reasons over employee requests, and triggers workflow actions such as drafting replies, logging requests, creating approval tasks, or escalating urgent security issues.

**Core idea:** Standard LLMs generate text. This system retrieves evidence first, uses specialized agents to reason over the evidence, and then routes the request to a business workflow.

## Business Problem and Project Value

HR teams frequently receive employee questions about sick leave, remote work, overtime, vacation requests, and security concerns. Manually reviewing and routing these requests can create delays, inconsistent answers, and missed urgent escalations.

This project uses Agentic RAG to retrieve relevant HR policy evidence, analyze an employee request, and recommend an appropriate workflow action.

Potential business value includes:

- Faster responses to routine employee questions
- More consistent policy guidance
- Reduced repetitive work for HR staff
- Improved documentation and request routing
- Faster escalation of urgent security incidents
- Human review when policy evidence is weak or uncertain

## Multi-Agent System Architecture

The solution uses three specialized agents and one coordinating team:

1. **HR Policy Retriever Agent** — retrieves evidence from the internal HR policy knowledge base.
2. **External Context Retriever Agent** — retrieves supporting HR automation and workflow context.
3. **Workflow Decision Agent** — recommends an appropriate workflow action, priority level, and reason.
4. **HR Policy Coordinator** — coordinates the agents and produces the final structured response.

The system demonstrates multi-agent coordination, retrieval from multiple knowledge sources, evidence-based reasoning, workflow routing, and human-review escalation.


## Environment Setup

Install the required Python libraries and configure the OpenAI API key securely before creating the knowledge sources and agents.

In [32]:
!pip install -q agno openai scikit-learn pandas numpy

In [ ]:
import os
import getpass

# Remove any previously stored key from the current notebook session
os.environ.pop("OPENAI_API_KEY", None)

# Enter the API key securely without displaying it
api_key = getpass.getpass("Enter your OpenAI API key: ").strip()
os.environ["OPENAI_API_KEY"] = api_key

print("OpenAI API key configured securely for this session.")

## Internal HR Policy Knowledge Base

This prototype uses five simulated internal HR policy documents covering sick leave, remote work, vacation, overtime, and data security.

The documents are transformed into TF-IDF vectors. When an employee submits a question, cosine similarity identifies the most relevant policy evidence. The source filename and similarity score are retained to make the response easier to trace and review.

In [35]:
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

hr_policy_docs = {
    "sick_leave_policy.txt": """
    Employees receive 8 paid sick days per calendar year.
    Sick leave may be used for illness, medical appointments, or caring for an immediate family member.
    Employees should notify their manager as soon as possible when taking sick leave.
    """,

    "remote_work_policy.txt": """
    Eligible employees may work remotely up to two days per week.
    Remote work requires manager approval before the remote work date.
    Employees must remain available during normal working hours and maintain productivity expectations.
    """,

    "vacation_policy.txt": """
    Vacation requests must be submitted at least two weeks in advance.
    Approval depends on staffing levels, business needs, and manager review.
    Employees should not finalize travel plans until vacation approval is confirmed.
    """,

    "overtime_policy.txt": """
    Overtime must be approved by a supervisor before extra hours are worked.
    Employees who work overtime without approval may be asked to provide justification.
    Approved overtime must be recorded in the timekeeping system.
    """,

    "data_security_policy.txt": """
    Employees must report suspected phishing emails, suspicious links, or possible data breaches immediately.
    Security incidents should be escalated to IT or the security team.
    Employees should not forward suspicious links or download unknown attachments.
    """
}

doc_names = list(hr_policy_docs.keys())
doc_texts = list(hr_policy_docs.values())

vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(doc_texts)

## Supporting Workflow Context

A second knowledge source provides supporting context about HR automation, remote work, leave administration, and cybersecurity escalation.

This information helps the agents understand common workflow practices. However, internal company policy always takes priority over supporting context when the coordinator prepares its final response.

In [36]:
external_context_docs = {
    "hr_automation_context.txt": """
    HR automation systems help reduce repetitive manual work by classifying employee requests,
    retrieving relevant policy information, routing approval cases, and escalating urgent issues.
    Common HR automation actions include drafting emails, creating manager approval tasks,
    logging requests, and sending notifications.
    """,

    "remote_work_context.txt": """
    Remote work workflows commonly require clear eligibility rules, manager approval,
    employee availability during normal working hours, and documented approval records.
    Automated systems can route remote work requests to managers for review.
    """,

    "cybersecurity_context.txt": """
    Security incident workflows require fast escalation when an employee reports phishing,
    suspicious links, data breaches, or possible account compromise. Automated alerts to IT
    or security teams can reduce response time and support incident tracking.
    """,

    "leave_policy_context.txt": """
    Leave policy workflows are commonly improved by automatic request logging, employee guidance,
    manager approval routing, and clear communication of available leave balances or policy rules.
    """
}

external_doc_names = list(external_context_docs.keys())
external_doc_texts = list(external_context_docs.values())

external_vectorizer = TfidfVectorizer(stop_words="english")
external_doc_matrix = external_vectorizer.fit_transform(external_doc_texts)

def retrieve_external_context(query: str, top_k: int = 2) -> str:
    """
    Retrieve supporting external/business context from a second knowledge source.
    This replaces live DuckDuckGo search to avoid DDGS runtime failures.
    """
    query_vector = external_vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, external_doc_matrix).flatten()

    top_indices = similarities.argsort()[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "source": external_doc_names[idx],
            "score": round(float(similarities[idx]), 3),
            "excerpt": external_context_docs[external_doc_names[idx]].strip()
        })

    return json.dumps(results, indent=2)

print("External context retriever created successfully.")

External context retriever created successfully.


## Agent Implementation and Coordination

The following section creates the three specialized agents and the HR Policy Coordinator.

Each agent has a defined role, tool access, and operating instructions. The coordinator combines their contributions into a structured, evidence-based response while prioritizing internal policy and recommending human review when evidence is weak or missing.

In [37]:
def decide_zapier_action(
    policy_evidence: str,
    external_context: str,
    employee_request: str
) -> str:
    """Determine the required workflow action using explicit business rules."""

    request = employee_request.lower()

    if any(phrase in request for phrase in [
        "phishing",
        "security incident",
        "suspicious link",
        "data breach",
        "account compromise"
    ]):
        decision = {
            "action": "Escalate to IT Security",
            "priority": "Urgent",
            "reason": "Employee reported a potential security incident."
        }

    elif any(phrase in request for phrase in [
        "remote work",
        "work remotely",
        "working remotely",
        "work from home",
        "wfh"
    ]):
        decision = {
            "action": "Create Remote Work Approval Task",
            "priority": "High",
            "reason": "Employee is requesting to work remotely."
        }

    elif any(phrase in request for phrase in [
        "sick day",
        "sick leave",
        "paid sick days"
    ]):
        decision = {
            "action": "Draft Sick Leave Email Reply",
            "priority": "Medium",
            "reason": "Employee is asking about sick leave policy."
        }

    else:
        decision = {
            "action": "Recommend HR Review",
            "priority": "Low",
            "reason": "No specific automation action was identified."
        }

    return json.dumps(decision, indent=2)

In [38]:
from agno.agent import Agent
from agno.team import Team
from agno.models.openai import OpenAIChat
import json

MODEL_ID = "gpt-4o-mini"


def retrieve_hr_policy(query: str, top_k: int = 2) -> str:
    """Retrieve relevant internal HR policy documents."""

    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, doc_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "source": doc_names[idx],
            "score": round(float(similarities[idx]), 3),
            "excerpt": hr_policy_docs[doc_names[idx]].strip()
        })

    return json.dumps(results, indent=2)


def decide_zapier_action(
    policy_evidence: str,
    external_context: str,
    employee_request: str
) -> str:
    """Determine the required workflow action using explicit business rules."""

    request = employee_request.lower()

    if any(phrase in request for phrase in [
        "phishing",
        "security incident",
        "suspicious link",
        "data breach",
        "account compromise"
    ]):
        decision = {
            "action": "Escalate to IT Security",
            "priority": "Urgent",
            "reason": "Employee reported a potential security incident."
        }

    elif any(phrase in request for phrase in [
        "remote work",
        "work remotely",
        "working remotely",
        "work from home",
        "wfh"
    ]):
        decision = {
            "action": "Create Remote Work Approval Task",
            "priority": "High",
            "reason": "Employee is requesting to work remotely."
        }

    elif any(phrase in request for phrase in [
        "sick day",
        "sick leave",
        "paid sick days"
    ]):
        decision = {
            "action": "Draft Sick Leave Email Reply",
            "priority": "Medium",
            "reason": "Employee is asking about sick leave policy."
        }

    else:
        decision = {
            "action": "Recommend HR Review",
            "priority": "Low",
            "reason": "No specific automation action was identified."
        }

    return json.dumps(decision, indent=2)


policy_agent = Agent(
    name="HR Policy Retriever Agent",
    role="Retrieves internal HR policy evidence",
    model=OpenAIChat(id=MODEL_ID, max_tokens=500),
    tools=[retrieve_hr_policy],
    instructions=[
        "Always retrieve internal HR policy evidence before answering.",
        "Use only the retrieved policy evidence.",
        "Always include the source filename.",
        "Do not introduce outside facts or assumptions.",
        "If evidence is insufficient, recommend human HR review."
    ],
    markdown=True,
)


context_agent = Agent(
    name="External Context Retriever Agent",
    role="Retrieves supporting business and workflow context",
    model=OpenAIChat(id=MODEL_ID, max_tokens=500),
    tools=[retrieve_external_context],
    instructions=[
        "Always use the supporting-context retrieval tool.",
        "Use only the retrieved supporting context.",
        "Do not override internal HR policy.",
        "Always include the source filename.",
        "Do not introduce outside facts or assumptions."
    ],
    markdown=True,
)


workflow_agent = Agent(
    name="Workflow Decision Agent",
    role="Recommends a Zapier-style workflow action",
    model=OpenAIChat(id=MODEL_ID, max_tokens=500),
    tools=[decide_zapier_action],
    instructions=[
        "Always use the deterministic workflow-routing tool.",
        "Return its action, priority, and reason without changing them.",
        "Do not claim that an external action was completed."
    ],
    markdown=True,
)


team = Team(
    name="HR Policy Coordinator",
    members=[policy_agent, context_agent, workflow_agent],
    model=OpenAIChat(id=MODEL_ID, max_tokens=1200),
    instructions=[
        "Coordinate the three specialized agents.",
        "Use only the supplied policy evidence and supporting context.",
        "Internal HR policy has priority over supporting context.",
        "Do not introduce outside facts, laws, statistics, or assumptions.",
        "Recommend human review when evidence is insufficient.",
        "Do not claim that a recommended workflow was completed.",
        """
        When requested by run_triage, return:
        1. Employee Request Classification
        2. Retrieved Policy Evidence
        3. Supporting Context
        4. Final HR Answer
        6. Business Value
        7. Risk or Failure Case

        Python inserts the validated workflow decision as Section 5.
        """
    ],
    show_members_responses=False,
)

print("Agents and HR Policy Coordinator created successfully.")

Agents and HR Policy Coordinator created successfully.


## Demonstration and Test Scenarios

The system is evaluated using three representative employee requests covering routine policy guidance, approval routing, and urgent security escalation. Each scenario is run separately to make its evidence, reasoning, and recommended workflow action easy to review.

In [39]:
from IPython.display import display, Markdown
import json

def run_triage(employee_request: str):
    """Run one request through the grounded multi-agent HR workflow."""

    print("=" * 90)
    print("EMPLOYEE REQUEST:", employee_request)
    print("=" * 90)

    try:
        policy_evidence = retrieve_hr_policy(employee_request, top_k=2)
        supporting_context = retrieve_external_context(employee_request, top_k=2)

        workflow_json = decide_zapier_action(
            policy_evidence=policy_evidence,
            external_context=supporting_context,
            employee_request=employee_request
        )
        workflow = json.loads(workflow_json)

        response = team.run(
            f"""
            Employee request:
            {employee_request}

            Retrieved internal policy evidence:
            {policy_evidence}

            Retrieved supporting context:
            {supporting_context}

            Produce these sections only:

            1. Employee Request Classification
            2. Retrieved Policy Evidence
            3. Supporting Context
            4. Final HR Answer
            6. Business Value
            7. Risk or Failure Case

            Do not create Section 5. Python will insert the validated
            workflow-routing decision.

            Use only the evidence and context supplied above. Do not introduce
            outside facts, statistics, laws, averages, or assumptions.
            """
        )

        content = response.content if hasattr(response, "content") else str(response)

        action_section = (
            "\n\n5. **Recommended Zapier Action:** "
            f"**{workflow['action']}**  \n"
            f"**Priority:** {workflow['priority']}  \n"
            f"**Reason:** {workflow['reason']}\n\n"
        )

        if "6." in content:
            content = content.replace("6.", action_section + "6.", 1)
        else:
            content += action_section

        display(Markdown(content))

    except Exception as error:
        print("TRIAGE FAILED")
        print(type(error).__name__)
        print(str(error))

### Scenario 1: Sick Leave Policy

This scenario tests whether the system retrieves the correct sick-leave policy and provides routine employee guidance without unnecessary escalation.

In [40]:
# Scenario 1: Routine sick-leave policy question
run_triage("How many paid sick days do employees receive each year?")

EMPLOYEE REQUEST: How many paid sick days do employees receive each year?


1. Employee Request Classification: Inquiry about sick leave policy.

2. Retrieved Policy Evidence: 
   - "Employees receive 8 paid sick days per calendar year. Sick leave may be used for illness, medical appointments, or caring for an immediate family member. Employees should notify their manager as soon as possible when taking sick leave." (Source: sick_leave_policy.txt)

3. Supporting Context: 
   - "Leave policy workflows are commonly improved by automatic request logging, employee guidance, manager approval routing, and clear communication of available leave balances or policy rules." (Source: leave_policy_context.txt)

4. Final HR Answer: Employees receive 8 paid sick days per calendar year.



5. **Recommended Zapier Action:** **Draft Sick Leave Email Reply**  
**Priority:** Medium  
**Reason:** Employee is asking about sick leave policy.

6. Business Value: Clear communication of the sick leave policy helps in employee satisfaction and ensures proper usage of sick days for health-related issues.

7. Risk or Failure Case: If employees are not aware of the sick leave policy, they may not utilize their entitled days properly, leading to health issues affecting productivity and attendance.

### Scenario 2: Remote Work Approval

This scenario tests whether the system retrieves the remote-work policy and recommends routing the employee’s request to a manager for approval.

In [41]:
# Scenario 2: Remote-work approval request
run_triage("Can I work remotely next Friday?")

EMPLOYEE REQUEST: Can I work remotely next Friday?


1. Employee Request Classification: Remote Work Request

2. Retrieved Policy Evidence: 
   - "Eligible employees may work remotely up to two days per week. Remote work requires manager approval before the remote work date. Employees must remain available during normal working hours and maintain productivity expectations." (remote_work_policy.txt)

3. Supporting Context: 
   - "Remote work workflows commonly require clear eligibility rules, manager approval, employee availability during normal working hours, and documented approval records. Automated systems can route remote work requests to managers for review." (remote_work_context.txt)

4. Final HR Answer: You may be eligible to work remotely next Friday; however, you will need to obtain manager approval prior to the remote work date. Please ensure you are available during normal working hours and maintain productivity expectations.



5. **Recommended Zapier Action:** **Create Remote Work Approval Task**  
**Priority:** High  
**Reason:** Employee is requesting to work remotely.

6. Business Value: Allowing remote work can enhance employee satisfaction and flexibility, leading to improved productivity and morale.

7. Risk or Failure Case: Without obtaining the necessary manager approval, the remote work request could be denied, and the employee might not be available at the office, affecting productivity and team collaboration.

### Scenario 3: Phishing Incident Escalation

This scenario tests whether the system recognizes an urgent security incident, retrieves the data-security policy, and recommends immediate escalation to the IT security team.

In [42]:
# Scenario 3: Urgent phishing incident
run_triage("I think I clicked a phishing link. What should I do?")

EMPLOYEE REQUEST: I think I clicked a phishing link. What should I do?


1. Employee Request Classification: Security incident reporting - phishing link clicked.

2. Retrieved Policy Evidence: 
   - Employees must report suspected phishing emails, suspicious links, or possible data breaches immediately. Security incidents should be escalated to IT or the security team. Employees should not forward suspicious links or download unknown attachments. (Source: data_security_policy.txt)

3. Supporting Context: 
   - Security incident workflows require fast escalation when an employee reports phishing, suspicious links, data breaches, or possible account compromise. Automated alerts to IT or security teams can reduce response time and support incident tracking. (Source: cybersecurity_context.txt)

4. Final HR Answer: Please report the suspected phishing link immediately to your supervisor or the IT/security team. Do not forward the suspicious link or download any attachments. Prompt reporting is essential for a quick response and to mitigate potential security risks.



5. **Recommended Zapier Action:** **Escalate to IT Security**  
**Priority:** Urgent  
**Reason:** Employee reported a potential security incident.

6. Business Value: Enhancing cybersecurity through prompt reporting of potential threats reduces the risk of data breaches and protects organizational assets.

7. Risk or Failure Case: Failing to report the phishing incident may lead to data breaches or security incidents, resulting in compromised sensitive information and potential operational disruptions.